In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Análisis y visualización de grandes conjuntos de datos

## Introducción a Graphistry
Graphistry es una plataforma GPU-acelerada para **visualizar y analizar grafos enormes** de forma interactiva.

### Casos de uso real
- Fraude financiero: detectar redes de operaciones sospechosas
- Redes sociales: analizar comunidades e influencers
- Supply Chain: rastrear dependencias complejas
- Epidemiología: modelar transmisión de enfermedades
- Ciberseguridad: detectar botnet y ataques coordinados

## Fundamentos teóricos de grafos

Antes de usar Graphistry, vamos a ver los fundamentos de un grafo y entender por qué estas estructuras son útiles para modelar relaciones.

In [ ]:
Image('/content/drive/MyDrive/Graph-Workshop/imgs/graph.png')



- Un **grafo** representa relaciones entre entidades.
- Los **nodos** son las entidades. Por ejemplo, personas, cuentas, hospitales, dispositivos, lugares, entre otros.
- Las **aristas** conectan nodos y pueden ser dirigidas o no dirigidas.
- Las aristas pueden tener **peso**, que mide intensidad, frecuencia o distancia.
- Los grafos ayudan a detectar **comunidades**, **puentes**, **rutas** y **puntos críticos**.

### Tipos más comunes
- **No dirigido**: la relación va en ambos sentidos.
- **Dirigido**: la dirección importa (A → B).
- **Ponderado**: cada conexión tiene un valor numérico.
- **Bipartito**: hay dos tipos de nodos y las conexiones solo ocurren entre tipos distintos.
- **Multigrafo**: pueden existir varias conexiones entre los mismos nodos.

### ¿Cuándo conviene pensar en grafos?
- Cuando el valor está en las **relaciones** y no solo en las filas (datos tabulares).
- Cuando necesitas encontrar **influencia** o **intermediación**.
- Cuando el problema tiene **propagación**: epidemias, rumores, fallos, fraudes.
- Cuando quieres ver **estructura global**: comunidades, hubs, cuellos de botella.

## Instalación y Setup

In [ ]:
libraries = ['networkx', 'matplotlib', 'pandas', 'numpy', 'scipy']

import subprocess
import sys
for lib in libraries:
    try:
        __import__(lib)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", lib, "-q"])

import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:
G_theory = nx.Graph()
G_theory.add_edges_from([
    ('A', 'B'),
    ('A', 'C'),
    ('B', 'D'),
    ('C', 'D'),
    ('D', 'E')
])

pos_theory = nx.spring_layout(G_theory, seed=23) # seed cambia la disposición de los nodos y nx.spring_layout es un algoritmo de disposición de nodos que simula un sistema de resortes para distribuir los nodos de manera equilibrada en el espacio.

fig, ax = plt.subplots(figsize=(5, 4))
nx.draw_networkx_nodes(G_theory, pos_theory, node_color='#0E6D54', node_size=1400, ax=ax)
nx.draw_networkx_edges(G_theory, pos_theory, width=2, alpha=0.6, edge_color='gray', ax=ax)
nx.draw_networkx_labels(G_theory, pos_theory, font_size=12, font_weight='bold', ax=ax)
ax.set_title('Grafo teórico mínimo', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

print('Nodos:', list(G_theory.nodes()))
print('Aristas:', list(G_theory.edges()))
print('Grados:', dict(G_theory.degree()))

### Representaciones del grafo
Un mismo grafo puede representarse de varias maneras:
- **Lista de adyacencia**: útil para navegación y algoritmos.
- **Matriz de adyacencia**: útil para álgebra lineal y visualización.
- **Lista de aristas**: útil para carga de datos y análisis tabular.

La elección depende del tamaño del problema y del tipo de análisis que quieras hacer.

In [ ]:
# Representaciones básicas del mismo grafo
adj_list = {node: list(G_theory.neighbors(node)) for node in G_theory.nodes()}

print('Lista de adyacencia:')
for node, neighbors in adj_list.items():
    print(f'  {node}: {neighbors}')

print('\nMatriz de adyacencia:')
adj_matrix = nx.to_pandas_adjacency(G_theory)
print(adj_matrix)

print('\nLista de aristas:')
for edge in G_theory.edges():
    print(f'  {edge}')

print('\nMétricas:')
print('  Componentes conectadas:', nx.number_connected_components(G_theory))
print('  Camino más corto A → E:', nx.shortest_path(G_theory, 'A', 'E'))
print('  Centralidad de grado:', nx.degree_centrality(G_theory))
# Centralidad de grado mide la importancia de un nodo en función de su número de conexiones.
# Un nodo con alta centralidad de grado tiene muchas conexiones directas con otros nodos, lo que puede indicar que es un punto clave en la red.

### Grafos dirigidos vs. Grafos no dirigidos

Hasta ahora usamos un grafo **no dirigido**, donde la relación vale en ambos sentidos. Eso sirve cuando la conexión es simétrica, como una amistad o una interacción mutua.

En un grafo **dirigido**, la flecha importa: A → B no implica B → A. Esto es útil para flujo de información, seguimiento, enlaces web o contagio en una sola dirección.

### Ejemplos rápidos
- **No dirigido**: amistad, coautoría, contacto mutuo.
- **Dirigido**: seguidores en redes, enlaces web, transmisión de una fuente a un receptor.
- **Ponderado**: intensidad de interacción, tiempo, distancia, frecuencia.

In [ ]:
G_undirected = nx.Graph()
G_undirected.add_edges_from([('A', 'B'), ('B', 'C'), ('C', 'D')])

G_directed = nx.DiGraph()
G_directed.add_edges_from([('A', 'B'), ('B', 'C'), ('C', 'A')])

print('No dirigido:')
print('  Edges:', list(G_undirected.edges()))
print('  Degrees:', dict(G_undirected.degree()))

print('\nDirigido:')
print('  Edges:', list(G_directed.edges()))
print('  In-degree:', dict(G_directed.in_degree()))
print('  Out-degree:', dict(G_directed.out_degree()))

fig, axes = plt.subplots(1, 2, figsize=(8, 3))

pos_u = nx.spring_layout(G_undirected, seed=2)
nx.draw_networkx(G_undirected, pos_u, ax=axes[0], node_color='#0E6D54', edge_color='gray', node_size=1400, with_labels=True)
axes[0].set_title('No dirigido')
axes[0].axis('off')

pos_d = nx.spring_layout(G_directed, seed=2)
nx.draw_networkx(G_directed, pos_d, ax=axes[1], node_color='#1f77b4', edge_color='gray', node_size=1400, with_labels=True, arrows=True)
axes[1].set_title('Dirigido')
axes[1].axis('off')

plt.tight_layout()
plt.show()

**Regla práctica:** si se puede preguntar “¿de quién a quién?”, usa dirigido; si solo importa están conectados, usa no dirigido.

### Grafos ponderados

Un grafo ponderado asigna un valor numérico a cada arista. Ese valor puede representar distancia, costo, duración, fuerza de relación o riesgo.

En epidemias, un peso puede representar **probabilidad de transmisión** o **intensidad de contacto**. En transporte, puede representar **tiempo de viaje**.

In [ ]:
G_weighted = nx.Graph()
G_weighted.add_edge('A', 'B', weight=2.5)
G_weighted.add_edge('B', 'C', weight=1.0)
G_weighted.add_edge('A', 'C', weight=4.0)
G_weighted.add_edge('C', 'D', weight=3.2)

for u, v, data in G_weighted.edges(data=True):
    print(f'{u} -- {v} | peso = {data["weight"]}')

fig, axes = plt.subplots(1, 1, figsize=(8, 3))

pos_w = nx.spring_layout(G_weighted, seed=2)
nx.draw_networkx(G_weighted, pos_w, ax=axes, node_color='#0E6D54', edge_color='gray', node_size=1400, with_labels=True)
axes.set_title('Ponderado')
axes.axis('off')
weighted_path = nx.shortest_path(G_weighted, 'A', 'D', weight='weight')
print('\nCamino mínimo ponderado A → D:', weighted_path)
print('Costo total:', nx.shortest_path_length(G_weighted, 'A', 'D', weight='weight'))

In [ ]:
print('Ruta mas corta de D a B sin peso:', nx.shortest_path(G_weighted, 'D', 'B'))
print('Costo total con peso:', nx.shortest_path_length(G_weighted, 'D', 'B', weight='weight'))

### Centralidad

Dos preguntas clave en análisis de grafos son:
- **¿Quién es importante?**  La centralidad permite encontrar nodos influyentes.

- **¿Qué grupos hay?**  Las comunidades ayudan a ver subgrupos fuertemente conectados dentro de la red.

In [ ]:
# Grado de centralidad es una medida de la importancia de un nodo en una red, basada en el número de conexiones que tiene con otros nodos. Un nodo con alta centralidad de grado es aquel que tiene muchas conexiones directas, lo que puede indicar que es un punto clave o influyente dentro de la red.
degree_centrality = nx.degree_centrality(G_theory)

# La centralidad de intermediación mide la importancia de un nodo en función de la cantidad de veces que actúa como puente a lo largo del camino más corto entre otros nodos. Un nodo con alta centralidad de intermediación es aquel que se encuentra en muchas rutas más cortas entre otros nodos, lo que puede indicar que es un punto crucial para la comunicación o el flujo de información dentro de la red.
betweenness_centrality = nx.betweenness_centrality(G_theory)

#Comunidades detectadas utilizando el algoritmo de modularidad codiciosa, que agrupa los nodos en comunidades basándose en la maximización de la modularidad, una medida que evalúa la densidad de conexiones dentro de las comunidades en comparación con las conexiones entre comunidades.
communities = list(nx.community.greedy_modularity_communities(G_theory))

fig, axes = plt.subplots(1, 1, figsize=(8, 3))

pos_w = nx.spring_layout(G_theory, seed=2)
nx.draw_networkx(G_theory, pos_w, ax=axes, node_color='#0E6D54', edge_color='gray', node_size=1400, with_labels=True)
axes.axis('off')
print('Centralidad de grado:')
for node, value in degree_centrality.items():
    print(f'  {node}: {value:.3f}')

print('\nCentralidad de intermediación:')
for node, value in betweenness_centrality.items():
    print(f'  {node}: {value:.3f}')

print('\nComunidades detectadas:')
for idx, community_nodes in enumerate(communities, start=1):
    print(f'  Comunidad {idx}: {sorted(list(community_nodes))}')


- **`degree_centrality`**  
    Mide qué tan conectado está cada nodo respecto al total posible de conexiones.  
    - Valor alto indica un nodo con muchas conexiones directas (más “activo” localmente).
    - En `G_theory`, **D** tiene el valor más alto, por eso actúa como hub local.

- **`betweenness_centrality`**  
    Mide cuántos caminos mínimos entre pares de nodos pasan por un nodo.  
    - Valor alto indica nodo “puente” o cuello de botella para el flujo en la red.
    - En `G_theory`, **D** también domina, indicando que conecta zonas del grafo.

- **`communities` (comunidades)**  
    Detecta grupos de nodos más conectados entre sí que con el resto del grafo.  
    - Sirve para segmentación, detección de módulos o subredes.
    - Se detectan dos grupos: `{'A','B','C'}` y `{'D','E'}`.

### Modelos de propagación epidémica

En un grafo, una epidemia puede verse como un proceso de expansión a través de aristas de contacto. Aquí usamos una versión muy simple para entender la idea:
- un nodo puede estar **susceptible**, **infectado** o **recuperado**;
- un nodo infectado puede contagiar a sus vecinos;
- la propagación depende de la estructura del grafo y de la probabilidad de transmisión.

Este modelo es una aproximación pedagógica, no una simulación epidemiológica completa.

> En este notebook se usa `infection_probability = 0.65` solo con fines didácticos para ejemplificar la dinámica de propagación, pero elegir este valor en la realialidad es una tarea más desafiante.


In [ ]:
np.random.seed(42)

COLOR = {'S': '#A8DADC', 'I': '#E63946', 'R': '#2DC653'}
LABEL = {'S': 'Susceptible', 'I': 'Infectado', 'R': 'Recuperado'}

state = {node: 'S' for node in G_theory.nodes()}
patient_zero = 'A'
state[patient_zero] = 'I'

infection_probability = 0.65
steps = []
state_steps = [state.copy()]

for _ in range(3):
    new_state = state.copy()
    newly_infected = []
    for node in G_theory.nodes():
        if state[node] == 'I':
            for neighbor in G_theory.neighbors(node):
                if state[neighbor] == 'S' and np.random.rand() < infection_probability:
                    new_state[neighbor] = 'I'
                    newly_infected.append(neighbor)
            new_state[node] = 'R'
    state = new_state
    steps.append((state.copy(), newly_infected))
    state_steps.append(state.copy())

print('Estado final por nodo:')
for node, node_state in state.items():
    print(f'  {node}: {node_state}')

print('\nDefinición de los estados:')
print('  S = susceptible')
print('  I = infectado')
print('  R = recuperado')

### Visualización de la propagación
Para que se entienda mejor, mostramos la evolución en paneles de tiempo:

- 🟦 **S**: Susceptible
- 🟥 **I**: Infectado
- 🟩 **R**: Recuperado

Se pueden ver las transisiones de contagio de un nodo a otro en el grafo.

In [ ]:
from matplotlib.patches import Patch

if 'state_steps' in globals() and len(state_steps) > 0:
    snapshots_vis = state_steps
else:
    initial_state = {node: 'S' for node in G_theory.nodes()}
    initial_state['A'] = 'I'
    snapshots_vis = [initial_state] + [snap for snap, _ in steps]

color_vis = {'S': '#5DADE2', 'I': '#E74C3C', 'R': '#58D68D'}
label_vis = {'S': 'Susceptible', 'I': 'Infectado', 'R': 'Recuperado'}

n_steps = len(snapshots_vis)
fig, axes = plt.subplots(1, n_steps, figsize=(3.3 * n_steps, 3.6))
if n_steps == 1:
    axes = [axes]

for t, ax in enumerate(axes):
    snap = snapshots_vis[t]
    node_colors = [color_vis[snap[node]] for node in G_theory.nodes()]
    nx.draw_networkx_edges(G_theory, pos_theory, ax=ax, edge_color='gray', alpha=0.45, width=2)
    nx.draw_networkx_nodes(G_theory, pos_theory, ax=ax, node_color=node_colors, node_size=1100)
    nx.draw_networkx_labels(G_theory, pos_theory, ax=ax, font_color='white', font_weight='bold')

    counts = {k: sum(v == k for v in snap.values()) for k in ['S', 'I', 'R']}
    ax.set_title(f"Paso {t}\nS:{counts['S']} I:{counts['I']} R:{counts['R']}", fontsize=10)
    ax.axis('off')

legend_items = [Patch(facecolor=color_vis[k], label=label_vis[k]) for k in ['S', 'I', 'R']]
fig.legend(handles=legend_items, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.08))
plt.suptitle('Propagación epidémica sobre el grafo', fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

La visualización del modelo SIR (Susceptible-Infectado-Recuperado) sobre un grafo pequeño permite  ilustrar cómo la estructura de las conexiones afecta la propagación de una epidemia.

- **Estados:**  
    - **S (Susceptible):** Nodo sano que puede infectarse.  
    - **I (Infectado):** Nodo que puede contagiar a sus vecinos.  
    - **R (Recuperado):** Nodo que ya no puede infectar ni ser infectado.

- **Simulación:**  
    1. Se parte de un nodo inicial infectado (“paciente cero”).
    2. En cada paso, los infectados pueden contagiar a sus vecinos susceptibles con cierta probabilidad (`infection_probability`).
    3. Los infectados pasan a recuperados en el siguiente paso.
    4. Se registra el estado de cada nodo en cada paso (`state_steps`).

- **Visualización:**  
    - Se muestran paneles con el grafo según el estado de cada nodo en cada paso.
    - Se grafica la evolución temporal del número de nodos en cada estado (curva SIR).

Esto permite observar cómo la epidemia se propaga, cuántos nodos se infectan y recuperan, y cómo la estructura de la red influye en la dinámica del contagio.

# Graphistry aplicado a un dataset de ciberseguridad


En esta sección, empleamos **Graphistry**, una plataforma de visualización de grafos acelerada por GPU. A diferencia de NetworkX o Matplotlib, Graphistry está pensada para exploración interactiva en el navegador: puedes hacer zoom, filtrar nodos en tiempo real y navegar grafos de millones de aristas sin perder fluidez.

El **dataset es `honeypot.csv`**, el cual contiene 220 alertas capturadas por un honeypot, un sistema diseñado para atraer y registrar intentos de intrusión. Cada fila es una combinación única de atacante, víctima, puerto y vulnerabilidad, con el número de intentos y el rango temporal del ataque.

El flujo que seguiremos es el proceso estándar de análisis con Graphistry:

1. **Registro:**  autenticación contra el servidor de renderizado  
2. **Exploración del dataset:**  entender la estructura antes de graficar  
3. **Grafo simple:** la representación más directa: `attackerIP → victimIP`  
4. **Hipergrafo:**  conectar todas las entidades presentes en cada alerta  
5. **Visualización avanzada:** codificar roles, intensidad y tiempo en el propio grafo

## 1. Registro

Graphistry renderiza los grafos en un servidor externo que aprovecha la GPU para calcular el layout y gestionar la interactividad.

Por ello, el siguiente paso es crear una cuenta gratuita en [hub.graphistry.com](https://hub.graphistry.com) y luego sustituir las credenciales en la celda siguiente. Si prefieres no registrarte ahora, puedes ejecutar todas las celdas de preparación de datos; únicamente las llamadas a `.plot()` requieren conexión activa (pero esta es la parte más atractiva).


In [ ]:
!pip install graphistry

In [ ]:
import graphistry

# Reemplaza con tus credenciales de hub.graphistry.com
graphistry.register(api=3, username='USER', password='PASSWOD',
                     protocol='https', server='hub.graphistry.com')
print("graphistry version:", graphistry.__version__)

## 2. Exploración del dataset

Antes de construir cualquier grafo conviene entender qué hay en los datos. El dataset agrupa alertas por combinación única de `(attackerIP, victimIP, victimPort, vulnName)`. Es decir, no son eventos individuales sino conteos agregados.

| Columna | Descripción |
|---------|-------------|
| `attackerIP` | IP de origen del ataque |
| `victimIP` | IP del sistema comprometido o expuesto |
| `victimPort` | Puerto de destino explotado |
| `vulnName` | Identificador de la vulnerabilidad utilizada |
| `count` | Número de intentos registrados para esa combinación |
| `time(max)` / `time(min)` | Ventana temporal del ataque en formato Unix timestamp |

**Desbalance del dataset**, alto contraste entre atacantes y víctimas.

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Graph-Workshop/data/honeypot.csv')
print(f"Filas   : {len(df)}")
print(f"Columnas: {list(df.columns)}\n")
df.sample(5)

In [ ]:
df['victimIP'].unique()

In [ ]:
df['attackerIP'].nunique()

In [ ]:
# Grafo simple, una arista por cada alerta attackerIP → victimIP
g = graphistry.edges(df, 'attackerIP', 'victimIP')
print(f"Aristas cargadas: {len(df)}")
df[['attackerIP', 'victimIP', 'vulnName', 'count']].head()

Grafo simple de ataque,  attackerIP  y victimIP

Primero, una representación simple, donde cada fila del dataset se convierte en una arista dirigida entre la IP atacante y la IP víctima. Se muestra la topología pura de quién ataca a quién.


In [ ]:
# renderizar en Graphistry (requiere credenciales del paso anterior)
g.plot()


**En el grafo anterior NO se puede apreciar la relación entre un atacante concreto y la vulnerabilidad que prefiere, o entre un puerto y las IPs que lo atacan.**

## 4. Hipergrafo: conectando todas las entidades

El **hipergrafo** de Graphistry resuelve esto convirtiendo cada fila en una conexión entre múltiples entidades a la vez. En lugar de una sola arista por alerta, obtenemos un nodo por cada entidad única (IP, puerto, vulnerabilidad) y aristas que describen las relaciones entre ellos.

Exploramos dos variantes:

**Approach 1, fila como nodo hub:** cada fila se convierte en un nodo central que conecta hacia sus entidades. Es más denso y permite ver patrones de co-ocurrencia.

**Approach 2, conexiones directas (`direct=True`):** se definen explícitamente qué pares de entidades se conectan. El resultado es más limpio y facilita preguntas del tipo *¿qué vulnerabilidades usa un atacante concreto?* o *¿qué puertos están asociados a una IP víctima?*

En ambos casos, `attackerIP` y `victimIP` se fusionan bajo un mismo tipo `ip` para evitar nodos duplicados cuando una misma dirección aparece en ambos roles.

In [ ]:
# Approach 1: cada fila es un nodo hub conectado a sus entidades
hg1 = graphistry.hypergraph(
    df,
    entity_types=['attackerIP', 'victimIP', 'victimPort', 'vulnName'],
    opts={
        # Fusionar attackerIP y victimIP en un solo tipo de nodo 'ip'
        'CATEGORIES': {
            'ip': ['attackerIP', 'victimIP']
        }
    }
)

hg1_g = hg1['graph']
print(f"Nodos : {len(hg1_g._nodes)}")
print(f"Aristas: {len(hg1_g._edges)}")
hg1_g.plot()

In [ ]:
# Approach 2: conexiones directas entre entidades (sin nodo fila intermedio)
hg2 = graphistry.hypergraph(
    df,
    entity_types=['attackerIP', 'victimIP', 'victimPort', 'vulnName'],
    direct=True,
    opts={
        'EDGES': {
            'attackerIP': ['victimIP', 'victimPort', 'vulnName'],
            'victimPort': ['victimIP'],
            'vulnName':   ['victimIP']
        },
        'CATEGORIES': {
            'ip': ['attackerIP', 'victimIP']
        }
    }
)

hg2_g = hg2['graph']
print(f"Nodos : {len(hg2_g._nodes)}")
print(f"Aristas: {len(hg2_g._edges)}")
hg2_g.plot()

 ##  Diferencia principal:
 - Approach 1 (fila como nodo hub): cada alerta crea un nodo intermedio (EventID) que conecta IP atacante, IP víctima, puerto y vulnerabilidad.
    - Más detalle por evento y mejor para co-ocurrencias, pero grafo más denso/ruidoso.

 - Approach 2 (direct=True): conecta entidades directamente (sin nodo intermedio visible).
   - Grafo más limpio y fácil de leer para relaciones directas, pero con menos énfasis en el contexto completo de cada fila.

## 5. Visualización avanzada: codificar el contexto en el grafo

Hasta aquí el grafo muestra *estructura*. En este paso añadimos *contexto*: queremos que la visualización responda preguntas sin necesidad de volver a la tabla.

Para eso construimos primero una tabla de nodos que asigna atributos a cada IP: su rol (atacante o víctima) y el total de intentos que ha generado o recibido. Esta tabla es la que **Graphistry** usará para controlar el estilo visual de cada nodo.

Las decisiones de codificación son deliberadas:

- **Color por rol:**  rojo para atacantes, blanco para víctimas. El contraste hace inmediata la lectura del grafo sin necesidad de leyenda.
- **Ícono por rol:**  `bomb` y `laptop` refuerzan el rol sin depender solo del color, útil cuando el grafo se imprime o se comparte en escala de grises.
- **Tamaño proporcional a `attacks`:**  los atacantes más activos ocupan más espacio visual, haciendo obvia la asimetría de actividad de un vistazo.
- **Color de arista por `time(min)`:**  el degradado azul → rojo mapea el tiempo del primer contacto registrado, permitiendo ver si los ataques se concentran en una ventana o están distribuidos en el tiempo.

Esta combinación de encodings convierte el grafo en un artefacto analítico: cada atributo visual tiene una razón, y el resultado se puede leer sin documentación adicional.

In [ ]:
import pandas as pd

# Tabla de víctimas
targets_df = (
    df[['victimIP']]
    .drop_duplicates()
    .rename(columns={'victimIP': 'node_id'})
    .assign(type='victim', attacks=0)
)

# Tabla de atacantes con conteo de ataques
attackers_df = (
    df.groupby('attackerIP')
    .agg(attacks=pd.NamedAgg(column='count', aggfunc='sum'))
    .reset_index()
    .rename(columns={'attackerIP': 'node_id'})
    .assign(type='attacker')
)

nodes_df = pd.concat([targets_df, attackers_df], ignore_index=True)

print(f"Atacantes únicos : {len(attackers_df)}")
print(f"Víctimas únicas  : {len(targets_df)}")
nodes_df.sort_values('attacks', ascending=False).head()

In [ ]:
g_adv = (
    graphistry
    .edges(df, 'attackerIP', 'victimIP')
    .nodes(nodes_df, 'node_id')

    # color por rol, atacante (rojo) vs víctima (green)
    .encode_point_color('type', categorical_mapping={
        'attacker': 'red',
        'victim':   'green'
    }, default_mapping='gray')

    # ícono por rol (FontAwesome v4)
    .encode_point_icon('type', categorical_mapping={
        'attacker': 'bomb',
        'victim':   'laptop'
    })

    # tamaño proporcional al número de ataques enviados
    .encode_point_size('attacks')

    # Color de arista, degradado temporal (azul → green= más reciente)
    .encode_edge_color('time(min)', palette=['blue', 'purple', 'red'], as_continuous=True)

    .addStyle(bg={'color': '#eee'}, page={'title': 'Honeypot — Red de Ataques'})
    .settings(url_params={'play': 1000, 'pointSize': 0.5})
)

g_adv.plot(as_files=False)

---

## Actividad práctica

Con el dataset cargado y los grafos construidos, la siguiente celda contiene código de exploración que puedes ejecutar, modificar y extender. Las preguntas están pensadas para recorrer las mismas operaciones que usarías en un análisis real: primero entender los datos en tablas, luego trasladar esas conclusiones al grafo.


1. ¿Qué IP concentra la mayor parte de los intentos? Suma `count` por `attackerIP` y observa la distribución ¿es uniforme o hay un atacante dominante?


In [ ]:
print(df.groupby('attackerIP')['count'].sum().sort_values(ascending=False).head())

2. ¿Qué vulnerabilidad aparece con más frecuencia? Revisa `vulnName` y considera si la concentración en pocas CVEs dice algo sobre el perfil del atacante.


In [ ]:
print(df['vulnName'].value_counts())

3. ¿Qué puerto es el objetivo más común? La respuesta tiene implicaciones directas sobre qué servicio está siendo apuntado

In [ ]:
#TODO

4. Vuelve a la celda de `g_adv` y cambia el campo de `.encode_edge_color(...)` de `time(min)` a `count`. ¿Cambia la lectura visual del grafo? ¿Qué información ganas y cuál pierdes?


In [ ]:
#TODO

5. Filtra el dataset a solo las filas con `count >= 10` y construye un nuevo grafo. ¿Qué atacantes desaparecen? ¿Qué estructura queda?

La celda siguiente incluye el código de partida para los puntos 1–3 y 5.

In [ ]:
df_top = df[df['count'] >= 10]
#TODO

## Lecturas adicionales

- [PyGraphistry en GitHub](https://github.com/graphistry/pygraphistry)
- [PyGraphistry — demos con bases de datos y APIs](https://github.com/graphistry/pygraphistry/tree/master/demos/demos_databases_apis)
- [graph-app-kit — dashboards con Streamlit](https://github.com/graphistry/graph-app-kit)
- [Guía de la UI de Graphistry](https://hub.graphistry.com/docs/ui/index/)
- [Documentación de encodings y estilos](https://pygraphistry.readthedocs.io/en/latest/)